<a href="https://colab.research.google.com/github/BenxiaHu/DeepLearning/blob/main/Deep_Learning_Pytorch_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# ---------- 0. 设备 ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- 1. 数据 ----------
class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

X = torch.randn(1000, 10)
true_w = torch.randn(10, 1)
y = X @ true_w + 0.1 * torch.randn(1000, 1)       # 造一个回归任务

dataset = MyDataset(X, y)
train_ds, val_ds = random_split(dataset, [800, 200])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)

# ---------- 2. 模型 ----------
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

model = MLP(10, 64, 1).to(device)

# ---------- 3. 损失 + 优化器 ----------
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ---------- 4. 训练循环 ----------
for epoch in range(20):
    # ---- 训练 ----
    model.train()                                  # 训练模式（Dropout/BN 起作用）
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)      # 数据搬到和模型同一设备

        pred = model(xb)                           # 前向
        loss = criterion(pred, yb)                 # 算损失

        optimizer.zero_grad()                      # 梯度清零
        loss.backward()                            # 反向传播
        optimizer.step()                           # 更新参数

        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_ds)

    # ---- 验证 ----
    model.eval()                                   # 评估模式
    val_loss = 0.0
    with torch.no_grad():                          # 不算梯度，省内存、更快
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_loss += criterion(model(xb), yb).item() * xb.size(0)
    val_loss /= len(val_ds)

    print(f"epoch {epoch+1:02d} | train {train_loss:.4f} | val {val_loss:.4f}")

# ---------- 5. 保存 / 加载 ----------
torch.save(model.state_dict(), "model.pt")
# model.load_state_dict(torch.load("model.pt")); model.eval()

In [ ]:
https://claude.ai/share/a77756b5-dbfd-4942-b969-20e86e463bb1
1. 数据：    Dataset → DataLoader
2. 模型：    class Net(nn.Module):  __init__ 定义层, forward 定义流向
3. 损失：    nn.MSELoss / nn.CrossEntropyLoss ...
4. 优化器：  torch.optim.Adam(model.parameters(), lr=...)
5. 循环：    model.train()
             for x, y in loader:
                 pred = model(x)
                 loss = criterion(pred, y)
                 optimizer.zero_grad()
                 loss.backward()
                 optimizer.step()
             model.eval() + torch.no_grad() 做验证

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # [B,1,28,28]  -> [B,16,28,28]
            nn.ReLU(),
            nn.MaxPool2d(2),                              #              -> [B,16,14,14]
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  #              -> [B,32,14,14]
            nn.ReLU(),
            nn.MaxPool2d(2),                              #              -> [B,32,7,7]
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                 #              -> [B,32*7*7]
            nn.Linear(32 * 7 * 7, num_classes),           #              -> [B,10]
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# 分类任务：criterion = nn.CrossEntropyLoss()
# 输出 logits 形状 [B, 10]，标签形状 [B]，dtype=torch.long